# 生成式AI可解释性在学习中的影响 - 实验数据分析报告

## 理论依据与业务目的
本分析旨在探讨**生成式AI的高可解释性特征（B组）**是否相比于**传统黑盒AI（A组）**，能更有效地促进学习者的深度学习。学习效果通过“概念图”的复杂性来衡量，具体指标为**节点数量的增长（node_growth）**和**关系连线数量的增长（edge_growth）**。

## 步骤 1：数据加载与预处理
**目的**：读取导出的 CSV 文件，清洗缺失值（如测试用户），并观察数据的基本统计特征。

**AI提示词 (Prompt)**：
> “请帮我用 pandas 读取 'experiment_analysis_data.csv' 文件，过滤掉 `node_growth` 为空的无效测试数据。然后分别统计 A组（控制组）和 B组（实验组）的用户数量，以及他们在节点增量 (`node_growth`) 和关系增量 (`edge_growth`) 上的均值和标准差。”

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据
df = pd.read_csv('d:/003作业ppt/03第二学期下/商业数据管理/experiment_analysis_data.csv')

# 数据清洗：去除空值（如前两行测试数据）
df_clean = df.dropna(subset=['node_growth', 'edge_growth']).copy()

# 基本描述性统计
desc_stats = df_clean.groupby('group_type')[['node_growth', 'edge_growth']].agg(['count', 'mean', 'std'])
print("基本描述性统计：")
print(desc_stats)

## 步骤 2：独立样本 T 检验 (Hypothesis Testing)
**目的**：通过统计学假设检验，验证 B 组在概念图增长上是否**显著**优于 A 组。

**AI提示词 (Prompt)**：
> “请使用 scipy.stats 对 A 组和 B 组的 `node_growth` 和 `edge_growth` 分别进行独立样本 t 检验。输出 t 值和 p 值，并解释结果在统计学上是否具有显著性差异（通常 p < 0.05 视为显著）。”

In [ ]:
group_a = df_clean[df_clean['group_type'] == 'A']
group_b = df_clean[df_clean['group_type'] == 'B']

# 对节点增量进行 T 检验
t_stat_node, p_val_node = stats.ttest_ind(group_a['node_growth'], group_b['node_growth'])
print(f"Node Growth T-test: t = {t_stat_node:.4f}, p = {p_val_node:.4e}")

# 对关系连线增量进行 T 检验
t_stat_edge, p_val_edge = stats.ttest_ind(group_a['edge_growth'], group_b['edge_growth'])
print(f"Edge Growth T-test: t = {t_stat_edge:.4f}, p = {p_val_edge:.4e}")

## 步骤 3：数据可视化
**目的**：直观展示两组在学习效果上的差异分布，为最终的网页报告生成图表。

**AI提示词 (Prompt)**：
> “请使用 seaborn 绘制两个箱线图（Boxplot），分别比较 A 组和 B 组在 `node_growth`（概念节点增量）和 `edge_growth`（概念关系增量）上的分布差异。要求图表美观，坐标轴有清晰的中文标签。”

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 节点增量箱线图
sns.boxplot(x='group_type', y='node_growth', data=df_clean, ax=axes[0], palette="Set2")
axes[0].set_title('A/B组 概念节点增量 (Node Growth) 比较', fontsize=14)
axes[0].set_xlabel('实验组别 (A=控制组, B=高可解释性组)', fontsize=12)
axes[0].set_ylabel('节点增量', fontsize=12)

# 关系连线增量箱线图
sns.boxplot(x='group_type', y='edge_growth', data=df_clean, ax=axes[1], palette="Set2")
axes[1].set_title('A/B组 概念关系连线增量 (Edge Growth) 比较', fontsize=14)
axes[1].set_xlabel('实验组别 (A=控制组, B=高可解释性组)', fontsize=12)
axes[1].set_ylabel('连线增量', fontsize=12)

plt.tight_layout()
plt.savefig('d:/003作业ppt/03第二学期下/商业数据管理/analysis_charts.png', dpi=300)
plt.show()